# Agent Trajectory Evaluation [Step 5 - Grading the Steps, Not Just the Answer]

> **MLCourse - Agentic AI - RAG Evaluation**

Notebooks 01-04 of this module evaluated **outputs**: was the answer faithful to
the context, was the context relevant, did a human judge like it. RAGAS, in
[`02_ragas_framework.ipynb`](02_ragas_framework.ipynb), does the same thing -
every one of its metrics takes `(question, contexts, answer, ground_truth)` and
scores the *result*.

For a RAG chain that is enough, because a chain has no interesting behaviour: it
retrieves once and generates once. For an **agent** it is badly insufficient,
because an agent *chooses what to do*. It picks tools, it orders them, it decides
when to stop, it recovers - or fails to recover - when a tool errors.

An agent that reaches the right answer after eleven redundant searches, two
timeouts and a lucky guess is **not a working agent**. Output evaluation calls it
a success. Trajectory evaluation is how you catch it.

### 1. What a trajectory is

A **trajectory** is the ordered record of everything the agent did:

```
  step 1  think   "I need the population figure first"
  step 2  tool    search_docs(query="population 2023")   -> 3 results
  step 3  tool    search_docs(query="population 2023")   -> 3 results   <- REDUNDANT
  step 4  tool    calculator("4.1e6 * 0.23")             -> ERROR: bad format
  step 5  tool    calculator("4100000 * 0.23")           -> 943000       <- RECOVERED
  step 6  answer  "About 943,000 people."
```

Output evaluation sees only the last line. Everything interesting - the duplicate
call, the malformed argument, the recovery - is in the lines above it.

### How this differs from RAGAS

| | RAGAS (output evaluation) | trajectory evaluation |
|---|---|---|
| input | question, contexts, answer, ground truth | the full ordered list of steps |
| question asked | *is the answer good?* | *was the process sound?* |
| catches | hallucination, irrelevant context, wrong answer | wrong tool, wrong order, redundant calls, no recovery, loops, runaway cost |
| misses | a correct answer reached by a terrible path | an elegant path to a wrong answer |
| when to use | any RAG pipeline | anything that chooses actions |

They are complementary and you want both. RAGAS tells you the answer was right;
trajectory evaluation tells you whether it will *keep* being right when the
inputs change slightly - because a process that succeeded by luck will not.

### 2. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


We use **Groq** with `qwen/qwen3.8-27b`. The free tier is roughly **8000 tokens
per minute**, so the judging loop below paces itself and backs off exponentially.
A local Ollama server at `localhost:11434` is the documented fallback -
`ChatOllama(model="llama3.1:8b")` swaps in with no other change.

In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


### 3. A small agent that actually runs

To evaluate trajectories we need real ones. We build a minimal tool-using agent
over the Alice corpus with three tools, and - importantly - we make it **record
every step**. Instrumenting for observability is not an afterthought; if your
agent does not emit a trajectory, you cannot evaluate one.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

raw_text = (DATA_DIR / "alice.txt").read_text(encoding="utf-8-sig")
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)

print(len(paragraphs), "paragraphs indexed")


# ---- the tools ------------------------------------------------------------
def search_docs(query):
    """Semantic search over the Alice corpus."""
    sims = doc_vectors @ encoder.encode([query], normalize_embeddings=True)[0]
    ids = np.argsort(sims)[::-1][:2]
    return " | ".join(f"[doc_{i}] {paragraphs[i][:200]}" for i in ids)


def count_word(word):
    """Count how many paragraphs mention a word."""
    return str(sum(1 for p in paragraphs if word.lower() in p.lower()))


def calculator(expression):
    """Evaluate a simple arithmetic expression."""
    if not re.fullmatch(r"[0-9\.\+\-\*/\(\)\s]+", expression or ""):
        raise ValueError(f"calculator only accepts arithmetic, got: {expression!r}")
    return str(eval(expression))          # safe: input is regex-restricted above


TOOLS = {"search_docs": search_docs, "count_word": count_word, "calculator": calculator}
print("tools:", list(TOOLS))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

237 paragraphs indexed
tools: ['search_docs', 'count_word', 'calculator']


In [4]:
TOOL_SPEC = (
    "search_docs(query) - semantic search over the book's text\n"
    "count_word(word) - how many paragraphs contain a word\n"
    "calculator(expression) - evaluate arithmetic like '12 * 3'"
)

AGENT_PROMPT = (
    "You are a research agent answering questions about 'Alice's Adventures in "
    "Wonderland'.\n\nAvailable tools:\n" + TOOL_SPEC + "\n\n"
    "Reply with EXACTLY ONE line, in one of these two forms:\n"
    "  ACTION: tool_name | argument\n"
    "  FINAL: your answer\n\n"
    "Use a tool only if you still need information. Never repeat a call you have "
    "already made.\n\nQuestion: {question}\n\nHistory so far:\n{history}\n\n"
    "Your single next line:"
)


def run_agent(question, max_steps=6):
    """Run the agent and RECORD every step - the trajectory is the point."""
    trajectory, history = [], "(nothing yet)"
    for step in range(1, max_steps + 1):
        line = ask(AGENT_PROMPT.format(question=question, history=history)).strip()
        line = line.splitlines()[0].strip()

        if line.upper().startswith("FINAL"):
            answer = line.split(":", 1)[1].strip()
            trajectory.append({"step": step, "type": "final", "answer": answer})
            return trajectory, answer

        match = re.match(r"ACTION:\s*(\w+)\s*\|\s*(.+)", line, re.IGNORECASE)
        if not match:
            trajectory.append({"step": step, "type": "malformed", "raw": line[:80]})
            history += f"\nstep {step}: malformed output, retrying"
            continue

        tool, arg = match.group(1), match.group(2).strip()
        record = {"step": step, "type": "tool", "tool": tool, "arg": arg}
        try:
            observation = TOOLS[tool](arg) if tool in TOOLS else f"no such tool: {tool}"
            record["ok"] = tool in TOOLS
            record["observation"] = observation[:200]
        except Exception as exc:
            record["ok"] = False
            record["observation"] = f"ERROR: {exc}"
        trajectory.append(record)
        history += f"\nstep {step}: {tool}({arg}) -> {record['observation'][:150]}"
        time.sleep(1.5)          # pacing for the free-tier token budget

    trajectory.append({"step": max_steps + 1, "type": "exhausted"})
    return trajectory, None


print("agent ready")

agent ready


### 4. Collect real trajectories

Three questions of increasing difficulty. We keep the raw trajectories - these
are the objects we are about to grade.

In [5]:
QUESTIONS = [
    "How many paragraphs mention the Cheshire Cat?",
    "What advice does the Caterpillar give Alice?",
    "How many paragraphs mention the Queen, multiplied by three?",
]

runs = []
for question in QUESTIONS:
    trajectory, answer = run_agent(question)
    runs.append({"question": question, "trajectory": trajectory, "answer": answer})
    print(f"\n=== {question}")
    for step in trajectory:
        if step["type"] == "tool":
            flag = "" if step.get("ok") else "   <- FAILED"
            print(f"  {step['step']}. {step['tool']}({step['arg'][:40]}){flag}")
            print(f"     -> {step['observation'][:90]}")
        elif step["type"] == "final":
            print(f"  {step['step']}. FINAL: {step['answer'][:100]}")
        else:
            print(f"  {step['step']}. [{step['type']}]")
    time.sleep(3.0)


=== How many paragraphs mention the Cheshire Cat?
  1. count_word(Cheshire Cat)
     -> 3
  2. FINAL: 3



=== What advice does the Caterpillar give Alice?
  1. search_docs(Caterpillar advice to Alice)
     -> [doc_91] Which brought them back again to the beginning of the conversation. Alice felt a 
  2. search_docs(Caterpillar "you are" advice)
     -> [doc_91] Which brought them back again to the beginning of the conversation. Alice felt a 
  3. search_docs(Caterpillar "you are")
     -> [doc_91] Which brought them back again to the beginning of the conversation. Alice felt a 
  4. search_docs(Caterpillar "you are" "I am")
     -> [doc_91] Which brought them back again to the beginning of the conversation. Alice felt a 
  5. FINAL: The Caterpillar advises Alice to "Keep your temper."



=== How many paragraphs mention the Queen, multiplied by three?
  1. count_word(Queen)
     -> 29
  2. calculator(29 * 3)
     -> 87
  3. FINAL: 87


### 5. Deterministic trajectory metrics

Before reaching for an LLM judge, compute what you can with plain code. These
metrics are free, exact, and stable across runs - which makes them the right
thing to put in CI.

- **Step count** - how much work was done.
- **Redundant calls** - the same tool with the same argument, twice. Pure waste.
- **Tool error rate** - fraction of calls that raised or returned an error.
- **Recovery** - after an error, did the *next* call succeed?
- **Completion** - did it produce a final answer, or exhaust its step budget?
- **Tool-choice appropriateness** - did it use the tool the task requires?

In [6]:
def trajectory_metrics(trajectory, expected_tools=None):
    tool_steps = [s for s in trajectory if s["type"] == "tool"]
    signatures = [(s["tool"], s["arg"].strip().lower()) for s in tool_steps]

    redundant = len(signatures) - len(set(signatures))
    errors = [i for i, s in enumerate(tool_steps) if not s.get("ok")]

    recovered = 0
    for i in errors:
        if i + 1 < len(tool_steps) and tool_steps[i + 1].get("ok"):
            recovered += 1

    used = {s["tool"] for s in tool_steps}
    metrics = {
        "steps": len(trajectory),
        "tool_calls": len(tool_steps),
        "redundant_calls": redundant,
        "tool_errors": len(errors),
        "error_rate": len(errors) / len(tool_steps) if tool_steps else 0.0,
        "recovery_rate": recovered / len(errors) if errors else None,
        "completed": any(s["type"] == "final" for s in trajectory),
        "malformed_outputs": sum(1 for s in trajectory if s["type"] == "malformed"),
    }
    if expected_tools:
        metrics["expected_tools_used"] = len(used & set(expected_tools)) / len(expected_tools)
        metrics["unexpected_tools"] = sorted(used - set(expected_tools))
    return metrics


EXPECTED = [["count_word"], ["search_docs"], ["count_word", "calculator"]]

for run, expected in zip(runs, EXPECTED):
    m = trajectory_metrics(run["trajectory"], expected)
    run["metrics"] = m
    print(f"\n{run['question']}")
    for key, value in m.items():
        print(f"   {key:<22} {value}")


How many paragraphs mention the Cheshire Cat?
   steps                  2
   tool_calls             1
   redundant_calls        0
   tool_errors            0
   error_rate             0.0
   recovery_rate          None
   completed              True
   malformed_outputs      0
   expected_tools_used    1.0
   unexpected_tools       []

What advice does the Caterpillar give Alice?
   steps                  5
   tool_calls             4
   redundant_calls        0
   tool_errors            0
   error_rate             0.0
   recovery_rate          None
   completed              True
   malformed_outputs      0
   expected_tools_used    1.0
   unexpected_tools       []

How many paragraphs mention the Queen, multiplied by three?
   steps                  3
   tool_calls             2
   redundant_calls        0
   tool_errors            0
   error_rate             0.0
   recovery_rate          None
   completed              True
   malformed_outputs      0
   expected_tools_used    1.0
  

### Reading these numbers

- **`redundant_calls > 0`** is the clearest waste signal there is, and it is
  purely mechanical to detect. It is usually caused by the agent not being shown
  its own history clearly enough.
- **`recovery_rate`** is the resilience metric. An agent that errors and then
  repeats the identical call is stuck in a loop; one that errors and then adjusts
  its argument is behaving well. `None` means there was nothing to recover from.
- **`expected_tools_used < 1.0`** means the agent skipped a tool the task needed
  - and it can still produce a plausible answer while doing so, which is exactly
  the failure output evaluation misses.
- **`completed == False`** means the step budget ran out. In production this is
  the metric that maps to cost.

### 6. LLM-as-judge for the parts code cannot check

Some questions are genuinely qualitative: was the *ordering* sensible? was a
search query well-formed for its goal? was stopping premature? For those, use a
judge - but give it a **rubric with named dimensions**, not "rate this out of
10".

In [7]:
JUDGE_PROMPT = (
    "You are evaluating the PROCESS an AI agent followed, not its final answer.\n\n"
    "Available tools:\n" + TOOL_SPEC + "\n\n"
    "Question the agent was given: {question}\n\n"
    "The agent's trajectory:\n{trajectory}\n\n"
    "Score each dimension from 1 (poor) to 5 (excellent):\n"
    "TOOL_CHOICE - did it pick the right tools for this task?\n"
    "ORDERING - were the steps in a sensible order, each building on the last?\n"
    "EFFICIENCY - was any step redundant or unnecessary?\n"
    "RECOVERY - if a tool failed, did it adapt sensibly? (score 5 if nothing failed)\n\n"
    "Reply in exactly this format, nothing else:\n"
    "TOOL_CHOICE: <n> - <short reason>\n"
    "ORDERING: <n> - <short reason>\n"
    "EFFICIENCY: <n> - <short reason>\n"
    "RECOVERY: <n> - <short reason>"
)


def render(trajectory):
    lines = []
    for s in trajectory:
        if s["type"] == "tool":
            status = "ok" if s.get("ok") else "FAILED"
            lines.append(f"step {s['step']}: {s['tool']}({s['arg']}) [{status}] "
                         f"-> {s['observation'][:110]}")
        elif s["type"] == "final":
            lines.append(f"step {s['step']}: FINAL ANSWER: {s['answer'][:150]}")
        else:
            lines.append(f"step {s['step']}: [{s['type']}]")
    return "\n".join(lines)


for run in runs:
    verdict = ask(JUDGE_PROMPT.format(question=run["question"],
                                      trajectory=render(run["trajectory"])))
    run["judge"] = verdict
    print("=" * 76)
    print(run["question"])
    print("=" * 76)
    print(verdict)
    print()
    time.sleep(4.0)          # pace the judge loop against the free-tier limit

How many paragraphs mention the Cheshire Cat?
TOOL_CHOICE: 5 - Used the specific `count_word` tool designed for this exact task.
ORDERING: 5 - Executed the tool call immediately followed by the final answer.
EFFICIENCY: 5 - Completed the task in the minimum number of steps (1 tool call).
RECOVERY: 5 - No tool failures occurred, so no recovery was needed.



What advice does the Caterpillar give Alice?
TOOL_CHOICE: 5 - search_docs was the correct tool for finding specific dialogue/advice in a text.
ORDERING: 3 - The agent started with a broad query and narrowed it down, which is logical, but the progression was slow and repetitive.
EFFICIENCY: 2 - Steps 2, 3, and 4 were largely redundant; the initial search in step 1 likely contained the necessary context, or a single refined search would have sufficed.
RECOVERY: 5 - No tools failed; the agent successfully retrieved documents in every step.



How many paragraphs mention the Queen, multiplied by three?
TOOL_CHOICE: 5 - Used count_word to find the frequency and calculator for the arithmetic, which are the exact tools needed.
ORDERING: 5 - Counted the word first, then used the result to perform the multiplication, a logical sequence.
EFFICIENCY: 5 - Only two steps were taken, both necessary, with no redundancy.
RECOVERY: 5 - No tools failed, so no recovery was needed.



### Parse the rubric back into numbers so it can be tracked over time.


In [ ]:
DIMENSIONS = ["TOOL_CHOICE", "ORDERING", "EFFICIENCY", "RECOVERY"]


def parse_scores(text):
    scores = {}
    for dim in DIMENSIONS:
        m = re.search(dim + r"\s*:\s*([1-5])", text)
        if m:
            scores[dim] = int(m.group(1))
    return scores


print(f"{'question':<44}" + "".join(f"{d[:9]:>11}" for d in DIMENSIONS))
print("-" * 90)
for run in runs:
    scores = parse_scores(run["judge"])
    run["scores"] = scores
    row = "".join(f"{scores.get(d, '-'):>11}" for d in DIMENSIONS)
    print(f"{run['question'][:42]:<44}{row}")


### 7. The point of the whole notebook: right answer, wrong process

Here is the demonstration that justifies trajectory evaluation existing. We
construct two trajectories that produce the **identical correct answer**, and
score both.

In [9]:
GOOD = [
    {"step": 1, "type": "tool", "tool": "count_word", "arg": "Queen",
     "ok": True, "observation": "34"},
    {"step": 2, "type": "tool", "tool": "calculator", "arg": "34 * 3",
     "ok": True, "observation": "102"},
    {"step": 3, "type": "final", "answer": "102"},
]

BAD = [
    {"step": 1, "type": "tool", "tool": "search_docs", "arg": "queen paragraphs count",
     "ok": True, "observation": "[doc_5] The Queen turned crimson with fury..."},
    {"step": 2, "type": "tool", "tool": "search_docs", "arg": "queen paragraphs count",
     "ok": True, "observation": "[doc_5] The Queen turned crimson with fury..."},
    {"step": 3, "type": "tool", "tool": "calculator", "arg": "thirty-four times three",
     "ok": False, "observation": "ERROR: calculator only accepts arithmetic"},
    {"step": 4, "type": "tool", "tool": "calculator", "arg": "thirty-four times three",
     "ok": False, "observation": "ERROR: calculator only accepts arithmetic"},
    {"step": 5, "type": "tool", "tool": "count_word", "arg": "Queen",
     "ok": True, "observation": "34"},
    {"step": 6, "type": "final", "answer": "102"},
]

QUESTION = "How many paragraphs mention the Queen, multiplied by three?"

print("Both trajectories end with the SAME answer: 102 (correct).")
print("Any output-only metric - RAGAS included - scores them identically.\n")

for label, traj in [("GOOD", GOOD), ("BAD", BAD)]:
    m = trajectory_metrics(traj, ["count_word", "calculator"])
    print(f"{label:<6} steps={m['steps']}  tool_calls={m['tool_calls']}  "
          f"redundant={m['redundant_calls']}  errors={m['tool_errors']}  "
          f"recovery={m['recovery_rate']}")

Both trajectories end with the SAME answer: 102 (correct).
Any output-only metric - RAGAS included - scores them identically.

GOOD   steps=3  tool_calls=2  redundant=0  errors=0  recovery=None
BAD    steps=6  tool_calls=5  redundant=2  errors=2  recovery=0.5


In [10]:
for label, traj in [("GOOD", GOOD), ("BAD", BAD)]:
    verdict = ask(JUDGE_PROMPT.format(question=QUESTION, trajectory=render(traj)))
    print("=" * 74)
    print(f"{label} trajectory - judge verdict")
    print("=" * 74)
    print(verdict)
    print()
    time.sleep(4.0)

GOOD trajectory - judge verdict
TOOL_CHOICE: 5 - Used count_word to find the frequency and calculator for the arithmetic, which are the exact tools needed.
ORDERING: 5 - Counted the word first, then used the result to perform the multiplication, a logical sequence.
EFFICIENCY: 5 - Only two steps were taken, both necessary, with no redundancy.
RECOVERY: 5 - No tools failed, so no recovery was needed.



BAD trajectory - judge verdict
TOOL_CHOICE: 4 - The agent eventually selected the correct tool (`count_word`) for the specific task, but initially relied on `search_docs` which is not designed for counting.
ORDERING: 3 - The agent attempted to calculate the final answer before obtaining the necessary count, and only used the correct counting tool after multiple failed attempts.
EFFICIENCY: 2 - The agent repeated the same `search_docs` query twice and the same invalid `calculator` expression twice, resulting in significant redundancy.
RECOVERY: 3 - The agent did not adapt to the calculator error by fixing the syntax (e.g., using `34 * 3`); instead, it repeated the exact same failing command before switching to a different tool.



The BAD trajectory used the wrong tool twice, made two identical calls, failed
twice on a malformed argument, never adapted that argument, and then stumbled on
the right tool - and it produced the correct final answer.

Every output metric in this module scores it **100%**. The deterministic
trajectory metrics and the judge both mark it down immediately. That gap is the
entire justification for this notebook.

### 8. Building a trajectory regression suite

The practical use is not a one-off report - it is a gate. Define expectations per
task, run them on every change to your prompts or tools, and fail the build when
the process degrades even if answers still look fine.

In [11]:
SUITE = [
    {"name": "counting task", "trajectory": GOOD,
     "expected_tools": ["count_word"],
     "limits": {"max_steps": 4, "max_redundant": 0, "max_errors": 0}},
    {"name": "arithmetic task (bad run)", "trajectory": BAD,
     "expected_tools": ["count_word", "calculator"],
     "limits": {"max_steps": 4, "max_redundant": 0, "max_errors": 0}},
]

print(f"{'case':<28}{'steps':>7}{'redund':>8}{'errors':>8}{'tools':>8}  verdict")
print("-" * 74)
failures = 0
for case in SUITE:
    m = trajectory_metrics(case["trajectory"], case["expected_tools"])
    lim = case["limits"]
    problems = []
    if m["steps"] > lim["max_steps"]:
        problems.append(f"steps>{lim['max_steps']}")
    if m["redundant_calls"] > lim["max_redundant"]:
        problems.append("redundant calls")
    if m["tool_errors"] > lim["max_errors"]:
        problems.append("tool errors")
    if m.get("expected_tools_used", 1.0) < 1.0:
        problems.append("missing expected tool")
    verdict = "PASS" if not problems else "FAIL: " + ", ".join(problems)
    failures += bool(problems)
    print(f"{case['name']:<28}{m['steps']:>7}{m['redundant_calls']:>8}"
          f"{m['tool_errors']:>8}{m.get('expected_tools_used', 1.0):>8.2f}  {verdict}")

print("-" * 74)
print(f"{len(SUITE) - failures}/{len(SUITE)} cases passed")

case                          steps  redund  errors   tools  verdict
--------------------------------------------------------------------------
counting task                     3       0       0    1.00  PASS
arithmetic task (bad run)         6       2       2    1.00  FAIL: steps>4, redundant calls, tool errors
--------------------------------------------------------------------------
1/2 cases passed


### 9. What to measure in production

| metric | why it matters | how |
|---|---|---|
| steps per task | direct cost and latency driver | deterministic |
| redundant call rate | pure waste; usually a prompt/memory bug | deterministic |
| tool error rate | broken tools or bad argument formatting | deterministic |
| recovery rate | resilience; low means loops | deterministic |
| completion rate | budget exhaustion | deterministic |
| tool-choice accuracy | is it solving the task the right way | deterministic if you declare expected tools |
| ordering quality | did each step build on the last | LLM judge |
| premature stopping | answered before gathering enough | LLM judge |

Start with the deterministic ones. They are free, exact, and catch the majority
of real agent regressions. Add the judge only for what code genuinely cannot
decide, and remember the judge is itself noisy - track its scores as trends, not
as absolute truth.

### 10. Where this connects

- [`01_evaluation_metrics.ipynb`](01_evaluation_metrics.ipynb) and
  [`02_ragas_framework.ipynb`](02_ragas_framework.ipynb) - output evaluation.
  Run both; they answer different questions.
- [`04_evaluation_pipeline.ipynb`](04_evaluation_pipeline.ipynb) - where these
  trajectory checks belong in an automated suite.
- [`../03_agentic_rag`](../03_agentic_rag/README.md) and
  [`../04_autonomous_rag`](../04_autonomous_rag/README.md) - the agents whose
  trajectories are worth grading in the first place.

### 11. Key takeaways

- RAGAS and every other output metric score the **answer**; an agent must also be
  judged on the **process**, because a correct answer reached badly will not stay
  correct.
- A trajectory is the ordered record of steps - and your agent must be
  instrumented to emit one before any of this is possible.
- Compute **deterministic** metrics first: step count, redundant calls, error
  rate, recovery rate, completion, tool-choice accuracy.
- Use an **LLM judge with a named rubric** only for the qualitative dimensions -
  ordering and premature stopping.
- The decisive test: two trajectories with the same correct answer can score
  identically on output metrics and very differently on process. Measure both.